In [1]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

OBS_PATH = PROJECT_ROOT / "data" / "processed" / "observations"
ACCOUNT_PATH = PROJECT_ROOT / "data" / "processed" / "accounts.parquet"

con = duckdb.connect()

print("Connected")

Connected


In [3]:
con.execute(f"""
    SELECT *
    FROM '{OBS_PATH}/*.parquet'
    LIMIT 5
""").df()

,organization,isp,asn,ip,port,transport,product,os,country_code,country_name,...,domains,hostnames,cpe,cpe23,http_status,http_server,http_title,http_host,http_components,timestamp
0,Meteverse Limited.,Meteverse Limited.,AS54994,138.113.61.106,15123,tcp,nginx,None,HK,Hong Kong,...,[],[],[cpe:/a:f5:nginx],[cpe:2.3:a:f5:nginx],400.0,nginx,400 Bad Request,138.113.61.106,"{'Nginx': {'categories': ['Web servers', 'Reve...",2026-09-14T09:59:27.083391
1,Korea Telecom,Korea Telecom,AS4766,218.150.120.133,8008,tcp,Chromecast,None,KR,"Korea, Republic of",...,[],[],[],[],403.0,NaN,NaN,218.150.120.133,"{'Nginx': None, 'jQuery': None, 'jQuery UI': N...",2026-09-14T09:59:56.544376
2,Google LLC,NaN,NaN,136.79.64.193,53,tcp,NaN,None,US,United States,...,[googleusercontent.com],[193.64.79.136.bc.googleusercontent.com],[],[],NaN,NaN,NaN,NaN,"{'Nginx': None, 'jQuery': None, 'jQuery UI': N...",2026-09-14T09:59:58.072744
3,"Fly.io, Inc.","Fly.io, Inc.",AS40509,137.66.8.132,12304,tcp,NaN,None,GB,United Kingdom,...,[flyio.net],[ip-137-66-8-132.customer.flyio.net],[],[],NaN,NaN,NaN,NaN,"{'Nginx': None, 'jQuery': None, 'jQuery UI': N...",2026-09-14T09:59:57.821438
4,Google LLC,"Level 3 Parent, LLC",AS3356,8.233.15.235,7171,tcp,NaN,None,US,United States,...,[googleusercontent.com],[235.15.233.8.bc.googleusercontent.com],[],[],NaN,NaN,NaN,NaN,"{'Nginx': None, 'jQuery': None, 'jQuery UI': N...",2026-09-14T09:59:57.212703


In [4]:
con.execute(f"""
    SELECT
        product,
        COUNT(*) AS observations
    FROM '{OBS_PATH}/*.parquet'
    WHERE product IS NOT NULL
      AND TRIM(product) != ''
    GROUP BY product
    ORDER BY observations DESC
    LIMIT 30
""").df()

,product,observations
0,nginx,105323
1,CloudFront httpd,40352
2,Apache httpd,35789
3,AWS ELB,20808
4,OpenSSH,16730
5,AkamaiGHost,14208
6,OpenResty,11174
7,Microsoft IIS httpd,9977
8,CloudFlare,7273
9,Socks4A,5169


In [5]:
con.execute(f"""
    SELECT
        port,
        COUNT(*) AS observations
    FROM '{OBS_PATH}/*.parquet'
    GROUP BY port
    ORDER BY observations DESC
    LIMIT 30
""").df()

,port,observations
0,80,252905
1,443,132127
2,3001,28532
3,5503,28374
4,1968,28009
5,7071,27821
6,1337,27807
7,14903,26835
8,3333,25009
9,20000,23781


In [6]:
con.execute(f"""
    SELECT
        cpe23,
        COUNT(*) AS observations
    FROM '{OBS_PATH}/*.parquet'
    WHERE cpe23 IS NOT NULL
    GROUP BY cpe23
    ORDER BY observations DESC
    LIMIT 30
""").df()

,cpe23,observations
0,[],1540991
1,[cpe:2.3:a:cloudflare:cloudflare],78332
2,[cpe:2.3:a:f5:nginx],76621
3,[cpe:2.3:a:imperva:incapsula],63008
4,[cpe:2.3:a:amazon:amazon_cloudfront],40301
5,[cpe:2.3:a:apache:http_server],22769
6,[cpe:2.3:a:amazon:elastic_load_balancing],22333
7,[cpe:2.3:a:fortinet:fortiweb],8577
8,"[cpe:2.3:a:f5:nginx, cpe:2.3:a:openresty:openr...",7285
9,"[cpe:2.3:a:f5:nginx:1.24.0, cpe:2.3:o:canonica...",5409


In [7]:
con.execute(f"""
    SELECT
        http_server,
        COUNT(*) AS observations
    FROM '{OBS_PATH}/*.parquet'
    WHERE http_server IS NOT NULL
      AND TRIM(http_server) != ''
    GROUP BY http_server
    ORDER BY observations DESC
    LIMIT 30
""").df()

,http_server,observations
0,cloudflare,82156
1,nginx,69553
2,CloudFront,40359
3,Apache,26534
4,awselb/2.0,21163
5,Tengine,20152
6,AkamaiGHost,14209
7,AmazonS3,9360
8,openresty,8391
9,Microsoft-IIS/10.0,7591


In [8]:
con.execute(f"""
    SELECT
        organization,
        COUNT(*) AS observations,
        COUNT(DISTINCT ip) AS unique_ips,
        COUNT(DISTINCT port) AS unique_ports,
        COUNT(DISTINCT product) AS unique_products,
        COUNT(DISTINCT country_code) AS countries_observed
    FROM '{OBS_PATH}/*.parquet'
    WHERE organization IS NOT NULL
      AND TRIM(organization) != ''
    GROUP BY organization
    ORDER BY unique_products DESC
    LIMIT 30
""").df()

,organization,observations,unique_ips,unique_ports,unique_products,countries_observed
0,Korea Telecom,20646,20537,576,175,1
1,"Chunghwa Telecom Co.,Ltd.",3157,3027,244,174,1
2,"Aliyun Computing Co., LTD",53656,44333,6233,133,1
3,Hetzner Online GmbH,9340,6967,374,107,7
4,"Comcast Cable Communications, LLC",1407,1260,163,102,3
5,Charter Communications Inc,1599,1534,148,101,1
6,Linode,12627,1994,752,98,15
7,Aliyun Computing Co.LTD,17744,15131,4301,97,4
8,"Amazon.com, Inc.",71144,29710,667,90,53
9,Microsoft Corporation,16632,9303,514,80,33


In [9]:
con.execute(f"""
    SELECT
        product,
        port,
        COUNT(*) AS observations
    FROM '{OBS_PATH}/*.parquet'
    WHERE product IS NOT NULL
       OR port IS NOT NULL
    GROUP BY product, port
    ORDER BY observations DESC
    LIMIT 100
""").df()

,product,port,observations
0,NaN,80,95288
1,NaN,443,74070
2,nginx,80,41375
3,CloudFront httpd,80,33309
4,NaN,3001,28393
...,...,...,...
95,Microsoft IIS httpd,443,2536
96,NaN,25001,2517
97,ntpd,123,2497
98,NaN,1023,2463


In [10]:
con.execute(f"""
    SELECT
        organization,
        COUNT(*) AS observations,
        COUNT(DISTINCT product) AS unique_products,
        COUNT(DISTINCT port) AS unique_ports,
        COUNT(DISTINCT ip) AS unique_ips
    FROM '{OBS_PATH}/*.parquet'
    WHERE organization IS NOT NULL
      AND TRIM(organization) != ''
    GROUP BY organization
    ORDER BY unique_products DESC
    LIMIT 50
""").df()

,organization,observations,unique_products,unique_ports,unique_ips
0,Korea Telecom,20646,175,576,20537
1,"Chunghwa Telecom Co.,Ltd.",3157,174,244,3027
2,"Aliyun Computing Co., LTD",53656,133,6233,44333
3,Hetzner Online GmbH,9340,107,374,6967
4,"Comcast Cable Communications, LLC",1407,102,163,1260
5,Charter Communications Inc,1599,101,148,1534
6,Linode,12627,98,752,1994
7,Aliyun Computing Co.LTD,17744,97,4301,15131
8,"Amazon.com, Inc.",71144,90,667,29710
9,Microsoft Corporation,16632,80,514,9303


In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

from src.signals.technology import classify_product

In [12]:
test_products = [
    "nginx",
    "OpenSSH",
    "VNC",
    "Postfix smtpd",
    "Squid http proxy",
    "MikroTik",
    "CloudFront httpd",
    "Something Unknown",
]

for product in test_products:
    print(f"{product:30} -> {classify_product(product)}")

nginx                          -> WEB
OpenSSH                        -> REMOTE_ACCESS
VNC                            -> REMOTE_ACCESS
Postfix smtpd                  -> EMAIL
Squid http proxy               -> PROXY
MikroTik                       -> NETWORK
CloudFront httpd               -> WEB
Something Unknown              -> OTHER


In [13]:
FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "account_features.parquet"

features = con.execute(f"""
    SELECT *
    FROM '{FEATURE_PATH}'
    LIMIT 20
""").df()

features

,organization,observation_count,unique_ips,unique_ports,unique_products,countries_observed,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count,first_seen,last_seen
0,"Aliyun Computing Co., LTD",53656,44333,6233,133,1,2510,5430,4,3413,82,2026-09-14T09:51:25.256936,2026-09-14T10:19:46.212008
1,"Akamai Technologies, Inc.",12080,10059,46,16,47,55,5,0,0,12,2026-09-14T09:55:11.959226,2026-09-14T10:14:01.312871
2,ACEVILLE PTE.LTD.,29651,22979,5117,26,34,302,63,0,10,0,2026-09-14T09:54:43.758346,2026-09-14T10:19:47.044941
3,*SE5-PTK*,634,605,396,7,1,1,1,0,0,0,2026-09-14T09:55:29.739279,2026-09-14T10:19:15.485357
4,BT-Central-Plus,584,583,13,8,1,1,3,0,0,0,2026-09-14T09:59:49.083764,2026-09-14T10:14:00.099768
5,A100 Row Inc,236,74,66,14,3,23,0,0,0,0,2026-09-14T09:58:22.441987,2026-09-14T10:13:59.690267
6,"Fly.io, Inc.",12875,8354,3433,2,7,0,0,0,2,0,2026-09-14T09:54:46.064036,2026-09-14T10:19:47.743020
7,WIND TRE S.P.A.,292,288,45,20,1,4,3,0,1,1,2026-09-14T09:55:58.321663,2026-09-14T10:13:56.834919
8,AFRINET-KINSHASA-1,1,1,1,0,1,0,0,0,0,0,2026-09-14T10:00:51.071258,2026-09-14T10:00:51.071258
9,"Liquid Web, L.L.C",448,338,23,15,1,254,3,3,0,0,2026-09-14T09:59:34.992235,2026-09-14T10:13:48.076756


In [14]:
con.execute(f"""
    SELECT
        organization,
        observation_count,
        unique_ips,
        unique_products,
        web_ip_count,
        remote_access_ip_count,
        email_ip_count,
        proxy_ip_count,
        network_ip_count
    FROM '{FEATURE_PATH}'
    ORDER BY remote_access_ip_count DESC
    LIMIT 20
""").df()

,organization,observation_count,unique_ips,unique_products,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count
0,"Aliyun Computing Co., LTD",53656,44333,133,2510,5430,4,3413,82
1,Aliyun Computing Co.LTD,17744,15131,97,1083,1657,1,1100,25
2,Google LLC,667710,248871,76,1439,797,5,59,2
3,Hetzner Online GmbH,9340,6967,107,2731,739,109,171,5
4,"DigitalOcean, LLC",6418,5095,77,2169,725,73,116,0
5,Linode,12627,1994,98,925,305,24,20,0
6,"Tencent cloud computing (Beijing) Co., Ltd.",1708,1611,51,573,283,0,19,0
7,Oracle Corporation,1461,1238,46,222,229,7,38,3
8,OVH SAS,4657,3959,76,1484,205,78,55,2
9,"Tencent Cloud Computing (Beijing) Co., Ltd",2408,2217,57,659,197,0,23,0


In [15]:
con.execute(f"""
    SELECT
        organization,
        observation_count,
        unique_ips,
        unique_products,
        web_ip_count,
        remote_access_ip_count
    FROM '{FEATURE_PATH}'
    ORDER BY web_ip_count DESC
    LIMIT 20
""").df()

,organization,observation_count,unique_ips,unique_products,web_ip_count,remote_access_ip_count
0,"Amazon.com, Inc.",71144,29710,90,23108,4
1,Amazon Technologies Inc.,25505,14335,78,4751,0
2,Meteverse Limited.,13397,4290,3,4057,0
3,METEVERSE LIMITED,17668,3877,3,3818,0
4,Hetzner Online GmbH,9340,6967,107,2731,739
5,"Cloudflare, Inc.",77211,18772,30,2705,1
6,"Aliyun Computing Co., LTD",53656,44333,133,2510,5430
7,"DigitalOcean, LLC",6418,5095,77,2169,725
8,A100 ROW GmbH,7582,6357,44,2103,0
9,IONOS SE,3475,3168,44,2001,136


In [16]:
from src.signals.organization import (
    normalize_organization,
    classify_organization,
)

In [17]:
test_organizations = [
    "Google LLC",
    "Amazon.com, Inc.",
    "Aliyun Computing Co., LTD",
    "Cloudflare, Inc.",
    "Korea Telecom",
    "DigitalOcean, LLC",
    "Hetzner Online GmbH",
    "Los Angeles Municipal Court",
    "Some Random Company",
]

In [18]:
for org in test_organizations:
    print(
        f"{org:50} -> "
        f"{normalize_organization(org):40} -> "
        f"{classify_organization(org)}"
    )

Google LLC                                         -> google llc                               -> UNKNOWN
Amazon.com, Inc.                                   -> amazoncom inc                            -> CLOUD_PROVIDER
Aliyun Computing Co., LTD                          -> aliyun computing co ltd                  -> CLOUD_PROVIDER
Cloudflare, Inc.                                   -> cloudflare inc                           -> CDN_SECURITY
Korea Telecom                                      -> korea telecom                            -> ISP
DigitalOcean, LLC                                  -> digitalocean llc                         -> CLOUD_PROVIDER
Hetzner Online GmbH                                -> hetzner online gmbh                      -> CLOUD_PROVIDER
Los Angeles Municipal Court                        -> los angeles municipal court              -> UNKNOWN
Some Random Company                                -> some random company                      -> UNKNOWN


In [19]:
ORG_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "account_features_v2.parquet"
)

con.execute(f"""
    SELECT
        organization_type,
        COUNT(*) AS organizations
    FROM '{ORG_FEATURE_PATH}'
    GROUP BY organization_type
    ORDER BY organizations DESC
""").df()

,organization_type,organizations
0,UNKNOWN,27802
1,ISP,5394
2,HOSTING_PROVIDER,639
3,CLOUD_PROVIDER,159
4,SECURITY_PROVIDER,40
5,CDN_SECURITY,31


In [20]:
con.execute(f"""
    SELECT
        organization,
        organization_type,
        unique_ips,
        unique_products,
        web_ip_count,
        remote_access_ip_count
    FROM '{ORG_FEATURE_PATH}'
    WHERE organization_type != 'UNKNOWN'
    ORDER BY unique_ips DESC
    LIMIT 30
""").df()

,organization,organization_type,unique_ips,unique_products,web_ip_count,remote_access_ip_count
0,Incapsula Inc,CDN_SECURITY,75602,15,88,0
1,"Aliyun Computing Co., LTD",CLOUD_PROVIDER,44333,133,2510,5430
2,"Amazon.com, Inc.",CLOUD_PROVIDER,29710,90,23108,4
3,Korea Telecom,ISP,20537,175,800,93
4,"Cloudflare, Inc.",CDN_SECURITY,18772,30,2705,1
5,Aliyun Computing Co.LTD,CLOUD_PROVIDER,15131,97,1083,1657
6,Amazon Technologies Inc.,CLOUD_PROVIDER,14335,78,4751,0
7,"Akamai Technologies, Inc.",CDN_SECURITY,10059,16,55,5
8,Microsoft Corporation,CLOUD_PROVIDER,9303,80,1697,189
9,Hetzner Online GmbH,CLOUD_PROVIDER,6967,107,2731,739


In [26]:
import importlib
import src.signals.organization as organization

importlib.reload(organization)

normalize_organization = organization.normalize_organization

In [28]:
test_organizations = [
    "Aliyun Computing Co., LTD",
    "Aliyun Computing Co.LTD",
    "Incapsula Inc",
    "Incapsula Inc.",
    "METEVERSE LIMITED",
    "Meteverse Limited.",
    "Google LLC",
    "Amazon.com, Inc.",
]

for org in test_organizations:
    print(f"{org:45} -> {normalize_organization(org)}")

Aliyun Computing Co., LTD                     -> aliyun computing
Aliyun Computing Co.LTD                       -> aliyun computing
Incapsula Inc                                 -> incapsula
Incapsula Inc.                                -> incapsula
METEVERSE LIMITED                             -> meteverse
Meteverse Limited.                            -> meteverse
Google LLC                                    -> google
Amazon.com, Inc.                              -> amazon com


In [30]:
con.execute("""
    SELECT COUNT(*) AS account_count
    FROM '../data/processed/account_features.parquet'
""").df()

,account_count
0,33615


In [32]:
con.execute("""
    SELECT
        normalized_organization,
        organization,
        observation_count,
        unique_ips,
        unique_products
    FROM '../data/processed/account_features.parquet'
    WHERE normalized_organization IN (
        'aliyun computing',
        'incapsula',
        'meteverse',
        'google'
    )
""").df()

,normalized_organization,organization,observation_count,unique_ips,unique_products
0,google,Google,1,1,0


In [34]:
con.execute("""
    SELECT COUNT(*) AS account_count
    FROM '../data/processed/account_features.parquet'
""").df()

,account_count
0,33615


In [46]:
con.execute("""
    SELECT *
    FROM '../data/processed/account_features.parquet'
    WHERE normalized_organization = 'google'
""").df()

,normalized_organization,organization,organization_type,observation_count,unique_ips,unique_ports,unique_products,countries_observed,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count,first_seen,last_seen
0,google,Google,UNKNOWN,667714,248875,17342,76,38,1439,797,5,59,2,2026-09-14T09:54:42.440496,2026-09-14T10:19:47.719416


In [47]:
con.execute("""
    SELECT *
    FROM '../data/processed/account_features.parquet'
    WHERE normalized_organization = 'aliyun computing'
""").df()

,normalized_organization,organization,organization_type,observation_count,unique_ips,unique_ports,unique_products,countries_observed,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count,first_seen,last_seen
0,aliyun computing,"Aliyun Computing Co, LTD",CLOUD_PROVIDER,71403,59465,7033,141,4,3593,7087,5,4513,107,2026-09-14T09:51:22.643245,2026-09-14T10:19:46.575483


In [40]:
con.execute("""
    SELECT
        organization,
        COUNT(*) AS observations
    FROM '../data/processed/observations/*.parquet'
    WHERE LOWER(TRIM(organization)) LIKE '%google%'
    GROUP BY organization
    ORDER BY observations DESC
""").df()

,organization,observations
0,Google LLC,667710
1,Google Fiber Inc.,68
2,ICG-1-GOOGLE,40
3,Google Asia Pacific Pte. Ltd. (GAPPL),32
4,Google Asia Pacific Pte. Ltd.,19
5,Google Cloud EMEA Ltd,6
6,IPACCT Global Google Cache,3
7,Google Argentina SRL,2
8,Google Inc.,2
9,GOOGLE CDN MNL3,1


In [42]:
con.execute("""
    SELECT
        organization,
        COUNT(*) AS observations
    FROM '../data/processed/observations/*.parquet'
    WHERE LOWER(TRIM(organization)) LIKE '%aliyun%'
    GROUP BY organization
    ORDER BY observations DESC
""").df()

,organization,observations
0,"Aliyun Computing Co., LTD",53656
1,Aliyun Computing Co.LTD,17744
2,"Aliyun Computing Co, LTD",3


In [44]:
con.execute("""
    SELECT COUNT(*) AS total_observations
    FROM '../data/processed/observations/*.parquet'
""").df()

,total_observations
0,2000000


In [49]:
con.execute("""
    SELECT
        COUNT(*) AS accounts,
        SUM(observation_count) AS observations,
        SUM(unique_ips) AS summed_unique_ips,
        MIN(first_seen) AS earliest_seen,
        MAX(last_seen) AS latest_seen
    FROM '../data/processed/account_features.parquet'
""").df()

,accounts,observations,summed_unique_ips,earliest_seen,latest_seen
0,33022,1996622.0,892592.0,2026-09-14T09:51:22.643245,2026-09-14T10:19:47.743020


In [51]:
con.execute("""
    SELECT
        organization_type,
        COUNT(*) AS accounts,
        SUM(observation_count) AS observations
    FROM '../data/processed/account_features.parquet'
    GROUP BY organization_type
    ORDER BY accounts DESC
""").df()

,organization_type,accounts,observations
0,UNKNOWN,27049,1044283.0
1,ISP,5154,113444.0
2,HOSTING_PROVIDER,619,11729.0
3,CLOUD_PROVIDER,140,310308.0
4,SECURITY_PROVIDER,36,4751.0
5,CDN_SECURITY,24,512107.0


In [52]:
con.execute("""
    SELECT
        normalized_organization,
        organization,
        organization_type,
        observation_count,
        unique_ips,
        web_ip_count,
        remote_access_ip_count,
        email_ip_count,
        proxy_ip_count,
        network_ip_count
    FROM '../data/processed/account_features.parquet'
    ORDER BY observation_count DESC
    LIMIT 20
""").df()

,normalized_organization,organization,organization_type,observation_count,unique_ips,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count
0,google,Google,UNKNOWN,667714,248875,1439,797,5,59,2
1,incapsula,Incapsula Inc,CDN_SECURITY,416563,77981,88,0,0,0,7
2,cloudflare,"CloudFlare, Inc.",CDN_SECURITY,77271,18832,2707,1,0,0,0
3,aliyun computing,"Aliyun Computing Co, LTD",CLOUD_PROVIDER,71403,59465,3593,7087,5,4513,107
4,amazon com,"Amazon.com, Inc.",CLOUD_PROVIDER,71144,29710,23108,4,0,38,5
5,meteverse,METEVERSE LIMITED,UNKNOWN,31065,8167,7875,0,0,0,1
6,aceville pte,ACEVILLE PTE.LTD.,UNKNOWN,29651,22979,302,63,0,10,0
7,amazon technologies,Amazon Technologies Inc.,CLOUD_PROVIDER,25505,14335,4751,0,2,87,2
8,internet rimon,Internet Rimon,UNKNOWN,21401,15663,0,1,0,0,0
9,korea telecom,Korea Telecom,ISP,20646,20537,800,93,5,3,213


In [57]:
con.execute("""
    SELECT
        technology_category,
        COUNT(*) AS product_rows
    FROM '../data/processed/product_mapping.parquet'
    GROUP BY technology_category
    ORDER BY product_rows DESC
""").df()

,technology_category,product_rows
0,OTHER,2012
1,WEB,8
2,REMOTE_ACCESS,4
3,PROXY,4
4,NETWORK,4
5,EMAIL,2
6,LOAD_BALANCER,1
7,UNKNOWN,1


In [58]:
con.execute("""
    SELECT
        technology_category,
        COUNT(*) AS product_rows
    FROM '../data/processed/product_mapping.parquet'
    GROUP BY technology_category
    ORDER BY product_rows DESC
""").df()

,technology_category,product_rows
0,OTHER,2012
1,WEB,8
2,REMOTE_ACCESS,4
3,PROXY,4
4,NETWORK,4
5,EMAIL,2
6,LOAD_BALANCER,1
7,UNKNOWN,1


In [59]:
con.execute("""
    SELECT
        organization_type,
        COUNT(*) AS accounts,
        AVG(unique_ips) AS avg_unique_ips,
        AVG(unique_ports) AS avg_unique_ports,
        AVG(unique_products) AS avg_unique_products,
        AVG(web_ip_count) AS avg_web_ips,
        AVG(remote_access_ip_count) AS avg_remote_access_ips,
        AVG(email_ip_count) AS avg_email_ips,
        AVG(proxy_ip_count) AS avg_proxy_ips,
        AVG(network_ip_count) AS avg_network_ips
    FROM '../data/processed/account_features.parquet'
    GROUP BY organization_type
    ORDER BY accounts DESC
""").df()

,organization_type,accounts,avg_unique_ips,avg_unique_ports,avg_unique_products,avg_web_ips,avg_remote_access_ips,avg_email_ips,avg_proxy_ips,avg_network_ips
0,UNKNOWN,27049,17.854264,4.648342,1.132722,1.819993,0.229879,0.046989,0.106732,0.101630
1,ISP,5154,19.923555,6.544043,1.968374,3.237291,0.403764,0.056849,0.109042,0.247769
2,HOSTING_PROVIDER,619,15.058158,3.067851,1.613893,6.775444,0.523425,0.450727,0.264943,0.016155
3,CLOUD_PROVIDER,140,1312.500000,145.685714,15.821429,425.050000,78.042857,3.714286,38.564286,1.435714
4,SECURITY_PROVIDER,36,82.416667,36.472222,0.527778,0.055556,0.027778,0.000000,0.000000,0.000000
5,CDN_SECURITY,24,4622.000000,91.000000,3.875000,120.625000,0.541667,0.000000,0.041667,1.000000


In [60]:
con.execute("""
    SELECT
        COUNT(*) AS accounts,

        SUM(CASE WHEN web_ip_count > 0
                 THEN 1 ELSE 0 END) AS accounts_with_web,

        SUM(CASE WHEN remote_access_ip_count > 0
                 THEN 1 ELSE 0 END) AS accounts_with_remote_access,

        SUM(CASE WHEN email_ip_count > 0
                 THEN 1 ELSE 0 END) AS accounts_with_email,

        SUM(CASE WHEN proxy_ip_count > 0
                 THEN 1 ELSE 0 END) AS accounts_with_proxy,

        SUM(CASE WHEN network_ip_count > 0
                 THEN 1 ELSE 0 END) AS accounts_with_network

    FROM '../data/processed/account_features.parquet'
""").df()

,accounts,accounts_with_web,accounts_with_remote_access,accounts_with_email,accounts_with_proxy,accounts_with_network
0,33022,11719.0,2914.0,875.0,896.0,2349.0


In [62]:
from src.signals.account_signals import build_account_signals

accounts = con.execute("""
    SELECT *
    FROM '../data/processed/account_features.parquet'
""").df()

accounts = build_account_signals(accounts)

accounts.head()

,normalized_organization,organization,organization_type,observation_count,unique_ips,unique_ports,unique_products,countries_observed,web_ip_count,remote_access_ip_count,...,first_seen,last_seen,has_web_exposure,has_remote_access_exposure,has_email_exposure,has_proxy_exposure,has_network_exposure,exposure_signal_count,technology_diversity,infrastructure_scale
0,jola cloud solutions,Jola Cloud Solutions Ltd,UNKNOWN,34,8,28,2,1,5,0,...,2026-09-14T10:00:07.655043,2026-09-14T10:12:54.526373,True,False,False,False,False,1,2,8
1,ntt america,NTT America,UNKNOWN,522,488,150,22,5,43,54,...,2026-09-14T09:54:55.413073,2026-09-14T10:18:28.387620,True,True,False,True,True,4,22,488
2,chinanet guangdong province network,CHINANET Guangdong Province Network,ISP,903,847,393,63,2,159,42,...,2026-09-14T09:54:54.093564,2026-09-14T10:19:09.900997,True,True,False,True,True,4,63,847
3,cox communications,Cox Communications,ISP,604,531,223,56,1,58,8,...,2026-09-14T09:55:23.352559,2026-09-14T10:17:25.072564,True,True,False,True,True,4,56,531
4,linode,Linode,CLOUD_PROVIDER,13863,2259,778,104,15,1062,351,...,2026-09-14T09:54:59.079557,2026-09-14T10:19:32.977986,True,True,True,True,False,4,104,2259


In [63]:
accounts[
    [
        "normalized_organization",
        "organization_type",
        "has_web_exposure",
        "has_remote_access_exposure",
        "has_email_exposure",
        "has_proxy_exposure",
        "has_network_exposure",
        "exposure_signal_count",
        "technology_diversity",
        "infrastructure_scale",
    ]
].head(20)

,normalized_organization,organization_type,has_web_exposure,has_remote_access_exposure,has_email_exposure,has_proxy_exposure,has_network_exposure,exposure_signal_count,technology_diversity,infrastructure_scale
0,jola cloud solutions,UNKNOWN,True,False,False,False,False,1,2,8
1,ntt america,UNKNOWN,True,True,False,True,True,4,22,488
2,chinanet guangdong province network,ISP,True,True,False,True,True,4,63,847
3,cox communications,ISP,True,True,False,True,True,4,56,531
4,linode,CLOUD_PROVIDER,True,True,True,True,False,4,104,2259
5,unyc sas,UNKNOWN,True,True,False,False,False,2,10,35
6,el roble,UNKNOWN,False,False,False,False,False,0,1,1
7,b2 net solutions,UNKNOWN,True,False,True,True,False,3,9,131
8,first server,UNKNOWN,True,True,False,False,False,2,4,18
9,hetzner online,CLOUD_PROVIDER,True,True,True,True,True,5,107,6967


In [65]:
con.execute("""
    SELECT
        pm.product,
        COUNT(*) AS observations
    FROM '../data/processed/observations/*.parquet' o
    JOIN '../data/processed/product_mapping.parquet' pm
        ON o.product = pm.product
    WHERE pm.technology_category = 'OTHER'
    GROUP BY pm.product
    ORDER BY observations DESC
    LIMIT 50
""").df()

,product,observations
0,AkamaiGHost,14208
1,Hikvision IP Camera,3040
2,Microsoft HTTPAPI httpd,2974
3,Chromecast,2830
4,ntpd,2497
5,ciscoSystems,2259
6,lighttpd,1566
7,Microsoft Azure Application Gateway,956
8,MySQL,828
9,Kubernetes,822


In [66]:
con.execute("""
    SELECT
        technology_category,
        COUNT(*) AS products
    FROM '../data/processed/product_mapping.parquet'
    GROUP BY technology_category
    ORDER BY products DESC
""").df()

,technology_category,products
0,OTHER,1976
1,WEB,14
2,REMOTE_ACCESS,8
3,IOT,8
4,NETWORK,7
5,PROXY,5
6,LOAD_BALANCER,3
7,DATABASE,3
8,CONTAINER_PLATFORM,3
9,SECURITY_NETWORK,2


In [67]:
con.execute("""
    SELECT
        product,
        technology_category
    FROM '../data/processed/product_mapping.parquet'
    WHERE product IN (
        'AkamaiGHost',
        'Hikvision IP Camera',
        'MySQL',
        'MariaDB',
        'PostgreSQL',
        'Kubernetes',
        'SonicWall',
        'Prometheus Node Exporter',
        'WinRM',
        'Portainer',
        'Dahua NVR',
        'Dahua XVR'
    )
    ORDER BY product
""").df()

,product,technology_category
0,AkamaiGHost,CDN
1,Dahua NVR,IOT
2,Dahua XVR,IOT
3,Hikvision IP Camera,IOT
4,Kubernetes,CONTAINER_PLATFORM
5,MariaDB,DATABASE
6,MySQL,DATABASE
7,Portainer,CONTAINER_PLATFORM
8,PostgreSQL,DATABASE
9,Prometheus Node Exporter,OBSERVABILITY


In [68]:
con.execute("""
    SELECT
        COUNT(*) AS accounts,

        SUM(CASE WHEN web_ip_count > 0 THEN 1 ELSE 0 END) AS web,

        SUM(CASE WHEN remote_access_ip_count > 0 THEN 1 ELSE 0 END) AS remote_access,

        SUM(CASE WHEN email_ip_count > 0 THEN 1 ELSE 0 END) AS email,

        SUM(CASE WHEN proxy_ip_count > 0 THEN 1 ELSE 0 END) AS proxy,

        SUM(CASE WHEN network_ip_count > 0 THEN 1 ELSE 0 END) AS network
    FROM '../data/processed/account_features.parquet'
""").df()

,accounts,web,remote_access,email,proxy,network
0,33022,12553.0,3102.0,875.0,896.0,3757.0


In [70]:
con.execute("""
    SELECT
        om.normalized_organization,

        COUNT(DISTINCT CASE
            WHEN pm.technology_category = 'DATABASE'
            THEN o.ip END
        ) AS database_ip_count,

        COUNT(DISTINCT CASE
            WHEN pm.technology_category = 'IOT'
            THEN o.ip END
        ) AS iot_ip_count,

        COUNT(DISTINCT CASE
            WHEN pm.technology_category = 'CONTAINER_PLATFORM'
            THEN o.ip END
        ) AS container_ip_count,

        COUNT(DISTINCT CASE
            WHEN pm.technology_category = 'SECURITY_NETWORK'
            THEN o.ip END
        ) AS security_network_ip_count,

        COUNT(DISTINCT CASE
            WHEN pm.technology_category = 'FILE_TRANSFER'
            THEN o.ip END
        ) AS file_transfer_ip_count

    FROM '../data/processed/observations/*.parquet' o

    JOIN '../data/processed/organization_mapping.parquet' om
        ON o.organization = om.organization

    LEFT JOIN '../data/processed/product_mapping.parquet' pm
        ON o.product = pm.product

    GROUP BY om.normalized_organization

    ORDER BY database_ip_count DESC
    LIMIT 20
""").df()

,normalized_organization,database_ip_count,iot_ip_count,container_ip_count,security_network_ip_count,file_transfer_ip_count
0,aliyun computing,75,0,11,65,29
1,tencent cloud computing (beijing),64,0,16,0,18
2,hetzner online,63,0,65,0,23
3,unified layer,61,0,0,0,45
4,digitalocean,53,0,16,0,12
5,google,47,0,509,0,0
6,microsoft,34,0,45,2,1
7,ovh sas,33,0,21,0,20
8,webhosting servers,31,0,0,0,0
9,bisecthosting,29,0,0,0,0


In [72]:
con.execute("""
    SELECT *
    FROM '../data/processed/account_features.parquet'
    LIMIT 1
""").df()

,normalized_organization,organization,organization_type,observation_count,unique_ips,unique_ports,unique_products,countries_observed,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count,database_ip_count,iot_ip_count,container_ip_count,security_network_ip_count,file_transfer_ip_count,first_seen,last_seen
0,amazon technologies,Amazon Technologies Inc.,CLOUD_PROVIDER,25505,14335,343,78,18,4856,0,2,87,2,1,36,60,2,0,2026-09-14T09:55:16.822453,2026-09-14T10:18:10.021540


In [73]:
con.execute("""
    SELECT
        normalized_organization,
        organization_type,
        observation_count,
        web_ip_count,
        remote_access_ip_count,
        email_ip_count,
        proxy_ip_count,
        network_ip_count,
        database_ip_count,
        iot_ip_count,
        container_ip_count,
        security_network_ip_count,
        file_transfer_ip_count
    FROM '../data/processed/account_features.parquet'
    WHERE normalized_organization IN (
        'aliyun computing',
        'google',
        'linode',
        'hetzner online',
        'korea telecom'
    )
""").df()

,normalized_organization,organization_type,observation_count,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count,database_ip_count,iot_ip_count,container_ip_count,security_network_ip_count,file_transfer_ip_count
0,aliyun computing,CLOUD_PROVIDER,71403,3754,7502,5,4760,107,75,0,11,65,29
1,linode,CLOUD_PROVIDER,13863,1100,351,25,21,0,17,2,22,118,4
2,hetzner online,CLOUD_PROVIDER,9340,2766,760,109,171,6,63,0,65,0,23
3,google,UNKNOWN,667714,1451,802,5,59,4,47,0,509,0,0
4,korea telecom,ISP,20646,1214,95,5,3,219,21,3090,2,1,1


In [74]:
from src.scoring.account_score import score_accounts

accounts = con.execute("""
    SELECT *
    FROM '../data/processed/account_features.parquet'
""").df()

scored_accounts = score_accounts(accounts)

In [75]:
scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "exposure_score",
        "technology_score",
        "infrastructure_score",
        "context_score",
    ]
].sort_values(
    "account_score",
    ascending=False
).head(20)

,normalized_organization,organization_type,account_score,exposure_score,technology_score,infrastructure_score,context_score
16413,korea telecom,ISP,93.99,40.0,25.0,13.985816,15
14403,139 162 0 0/16,UNKNOWN,88.42,37.0,22.5,8.923028,20
16410,chunghwa telecom,ISP,87.18,37.0,22.5,12.675497,15
16443,internet utilities europe and asia,UNKNOWN,86.93,35.0,20.0,11.928773,20
2,aliyun computing,CLOUD_PROVIDER,86.50,37.0,22.5,15.000000,12
20492,oracle,UNKNOWN,86.26,35.0,20.0,11.259093,20
10,microsoft,CLOUD_PROVIDER,85.33,37.0,22.5,13.830026,12
6161,google,UNKNOWN,85.26,33.0,17.5,14.764847,20
6174,v tal,UNKNOWN,85.15,35.0,20.0,10.148437,20
6230,scaleway,UNKNOWN,84.96,35.0,20.0,9.963723,20


In [77]:
candidates = con.execute("""
    SELECT
        normalized_organization,
        organization,
        organization_type,
        observation_count,
        unique_ips,
        unique_ports,
        unique_products,
        web_ip_count,
        remote_access_ip_count,
        email_ip_count,
        proxy_ip_count,
        network_ip_count,
        database_ip_count,
        iot_ip_count,
        container_ip_count,
        security_network_ip_count,
        file_transfer_ip_count
    FROM '../data/processed/account_features.parquet'
    WHERE unique_ips > 0
    ORDER BY RANDOM()
    LIMIT 30
""").df()

candidates

,normalized_organization,organization,organization_type,observation_count,unique_ips,unique_ports,unique_products,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count,database_ip_count,iot_ip_count,container_ip_count,security_network_ip_count,file_transfer_ip_count
0,macweb com,MACWEB.COM,UNKNOWN,3,3,3,2,1,0,0,0,0,0,0,0,0,0
1,ejie,EJIE,UNKNOWN,20,3,2,0,0,0,0,0,0,0,0,0,0,0
2,ligne web services sarl,Ligne Web Services SARL,UNKNOWN,2,2,2,0,0,0,0,0,0,0,0,0,0,0
3,adesso as a service,adesso as a service GmbH,UNKNOWN,1,1,1,0,0,0,0,0,0,0,0,0,0,0
4,pt buana visualnet sentra,PT Buana Visualnet Sentra,UNKNOWN,2,2,2,2,0,0,0,0,1,0,0,0,0,0
5,parabola d o o,Parabola D.O.O.,UNKNOWN,1,1,1,1,1,0,0,0,0,0,0,0,0,0
6,yamaichi,yamaichi,UNKNOWN,1,1,1,0,0,0,0,0,0,0,0,0,0,0
7,net-dynamicadsl-jkt,NET-DYNAMICADSL-JKT,UNKNOWN,1,1,1,1,1,0,0,0,0,0,0,0,0,0
8,on assignment,"On Assignment, Inc.",UNKNOWN,1,1,1,1,1,0,0,0,0,0,0,0,0,0
9,cits,CITS Private Limited,UNKNOWN,1,1,1,1,0,1,0,0,0,0,0,0,0,0


In [78]:
high_score = scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
    ]
].sort_values(
    "account_score",
    ascending=False
).head(10)

high_score

,normalized_organization,organization_type,account_score,observation_count,unique_ips,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count
16413,korea telecom,ISP,93.99,20646,20537,1214,95,21,3090
14403,139 162 0 0/16,UNKNOWN,88.42,919,153,56,38,1,0
16410,chunghwa telecom,ISP,87.18,3157,3027,603,76,1,258
16443,internet utilities europe and asia,UNKNOWN,86.93,2159,1946,93,37,2,0
2,aliyun computing,CLOUD_PROVIDER,86.50,71403,59465,3754,7502,75,0
20492,oracle,UNKNOWN,86.26,1489,1265,235,232,6,0
10,microsoft,CLOUD_PROVIDER,85.33,17946,10379,2563,224,34,0
6161,google,UNKNOWN,85.26,667714,248875,1451,802,47,0
6174,v tal,UNKNOWN,85.15,514,464,37,13,1,1
6230,scaleway,UNKNOWN,84.96,3650,410,68,27,3,0


In [79]:
medium_score = scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
    ]
].query(
    "account_score >= 45 and account_score <= 65"
).sample(
    min(10, len(scored_accounts))
)

medium_score

,normalized_organization,organization_type,account_score,observation_count,unique_ips,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count
23149,ucloud,UNKNOWN,49.44,36,34,7,4,0,0
12329,h4y technologies,UNKNOWN,57.97,138,132,22,6,0,0
2956,serverfreak technologies sdn bhd,UNKNOWN,46.43,17,13,7,0,0,0
16518,vodafone ono s a,ISP,45.79,101,101,11,2,0,0
12647,madgenius com,UNKNOWN,57.08,18,17,4,1,0,0
52,netrouting,UNKNOWN,49.40,9,9,3,1,1,0
23379,lnaub658 aubervilliers,UNKNOWN,47.41,8,8,1,1,0,1
15066,long van system solution jsc,UNKNOWN,49.32,8,8,5,1,0,0
13072,friendhosting,HOSTING_PROVIDER,48.52,22,22,8,1,0,0
2510,hostdime com,UNKNOWN,54.14,313,195,25,0,1,0


In [80]:
low_score = scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
    ]
].sort_values(
    "account_score"
).head(10)

low_score

,normalized_organization,organization_type,account_score,observation_count,unique_ips,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count
25746,zscaler vienna,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
26407,fortinet technologies india pvt,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
23213,zscaler paris,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
25798,zscaler sao paulo,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
30313,zscaler stockholm,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
25935,zscaler ams2,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
25085,zscaler brussels,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
21791,zscaler marseille,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
18582,zscaler singapore,SECURITY_PROVIDER,10.85,1,1,0,0,0,0
1520,zscaler helsinki,SECURITY_PROVIDER,10.85,1,1,0,0,0,0


In [82]:
providers = scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
    ]
]

providers = providers[
    providers["organization_type"].isin([
        "CLOUD_PROVIDER",
        "CDN_SECURITY",
        "ISP",
        "HOSTING_PROVIDER",
        "SECURITY_PROVIDER",
    ])
]

providers = providers.sort_values(
    "account_score",
    ascending=False
).head(10)

providers

,normalized_organization,organization_type,account_score,observation_count,unique_ips,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count
16413,korea telecom,ISP,93.99,20646,20537,1214,95,21,3090
16410,chunghwa telecom,ISP,87.18,3157,3027,603,76,1,258
2,aliyun computing,CLOUD_PROVIDER,86.50,71403,59465,3754,7502,75,0
10,microsoft,CLOUD_PROVIDER,85.33,17946,10379,2563,224,34,0
16439,telecom italia s p a,ISP,84.95,713,692,50,6,1,21
2048,linode,CLOUD_PROVIDER,84.50,13863,2259,1100,351,17,2
6,deutsche telekom ag,ISP,84.12,2904,2845,322,22,0,13
6184,asia pacific network information center pty,ISP,82.50,3084,2699,846,134,2,0
12334,jsc ukrtelecom,ISP,81.68,1917,1834,29,6,2,3
51,asia pacific network information centre,ISP,80.56,554,478,79,26,6,1


In [83]:
low_pool = scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
    ]
].query("account_score <= 25")

low_score = low_pool.sample(
    min(10, len(low_pool)),
    random_state=42
)

low_score

,normalized_organization,organization_type,account_score,observation_count,unique_ips,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count
10053,site - soluc em inf telec e eng ltda - brsite,UNKNOWN,21.45,1,1,0,0,0,0
19574,cloudnet cable and telecoms,ISP,15.85,1,1,0,0,0,0
4122,reassign to intriplex ( thailand ),UNKNOWN,21.45,1,1,0,0,0,0
32983,rural telecom sl,ISP,15.85,1,1,0,0,0,0
9537,canaa telecomunicações ltda - me,ISP,16.76,2,2,0,0,0,0
23966,entplexit,UNKNOWN,20.85,1,1,0,0,0,0
19942,all about packaging,UNKNOWN,20.85,1,1,0,0,0,0
29388,elon group ab,UNKNOWN,20.85,1,1,0,0,0,0
5436,liuzhou city jingying netbar,UNKNOWN,20.85,1,1,0,0,0,0
19394,080878 smif,UNKNOWN,20.85,1,1,0,0,0,0


In [84]:
security_providers = scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
    ]
]

security_providers = security_providers[
    security_providers["organization_type"] == "SECURITY_PROVIDER"
].sort_values(
    "account_score",
    ascending=False
)

security_providers.head(10)

,normalized_organization,organization_type,account_score,observation_count,unique_ips,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count
2109,palo alto networks,SECURITY_PROVIDER,27.13,1538,1236,1,0,0,0
14388,zscaler,SECURITY_PROVIDER,26.60,2557,1326,1,0,0,0
23728,fortinet,SECURITY_PROVIDER,24.66,25,25,0,1,0,0
12406,zscaler australia pty,SECURITY_PROVIDER,15.32,76,48,0,0,0,0
12497,zscaler softech india,SECURITY_PROVIDER,15.14,268,145,0,0,0,0
15242,zscaler osaka,SECURITY_PROVIDER,14.36,50,31,0,0,0,0
12749,zscaler switzerland,SECURITY_PROVIDER,14.12,14,13,0,0,0,0
14838,zscaler softech india private limited - hyderabad,SECURITY_PROVIDER,13.84,66,39,0,0,0,0
15646,zscaler tokyo,SECURITY_PROVIDER,13.74,24,14,0,0,0,0
22586,zscaler amsterdam ams2,SECURITY_PROVIDER,13.62,35,15,0,0,0,0


In [86]:
unknown_security = scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
        "container_ip_count",
    ]
]

unknown_security = unknown_security[
    (unknown_security["organization_type"] == "UNKNOWN") &
    (
        (unknown_security["remote_access_ip_count"] > 0) |
        (unknown_security["database_ip_count"] > 0) |
        (unknown_security["iot_ip_count"] > 0) |
        (unknown_security["container_ip_count"] > 0)
    )
].sort_values(
    "account_score",
    ascending=False
)

unknown_security.head(15)

,normalized_organization,organization_type,account_score,observation_count,unique_ips,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count,container_ip_count
14403,139 162 0 0/16,UNKNOWN,88.42,919,153,56,38,1,0,1
16443,internet utilities europe and asia,UNKNOWN,86.93,2159,1946,93,37,2,0,2
20492,oracle,UNKNOWN,86.26,1489,1265,235,232,6,0,17
6161,google,UNKNOWN,85.26,667714,248875,1451,802,47,0,509
6174,v tal,UNKNOWN,85.15,514,464,37,13,1,1,0
6230,scaleway,UNKNOWN,84.96,3650,410,68,27,3,0,40
16790,tt dotcom sdn bhd,UNKNOWN,83.16,119,110,9,4,1,4,0
22529,huawei public cloud service (huawei software t...,UNKNOWN,80.79,1580,1198,114,75,9,0,0
16435,internet utilities na,UNKNOWN,80.50,501,486,108,19,1,0,1
16544,ee,UNKNOWN,80.33,606,540,125,92,2,2,2


In [87]:
eval_candidates = scored_accounts[
    scored_accounts["normalized_organization"].isin([
        "korea telecom",
        "139 162 0 0/16",
        "chunghwa telecom",
        "internet utilities europe and asia",
        "aliyun computing",
        "oracle",
        "microsoft",
        "google",
        "ucloud",
        "h4y technologies",
        "serverfreak technologies sdn bhd",
        "vodafone ono s a",
        "madgenius com",
        "friendhosting",
        "hostdime com",
        "site - soluc em inf telec e eng ltda - brsite",
        "cloudnet cable and telecoms",
        "reassign to intriplex ( thailand )",
        "rural telecom sl",
        "entplexit",
        "palo alto networks",
        "zscaler",
        "fortinet",
        "zscaler australia pty",
        "zscaler softech india",
        "v tal",
        "scaleway",
        "tt dotcom sdn bhd",
        "huawei public cloud service (huawei software t...",
        "peg tech"
    ])
]

eval_candidates[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "unique_ports",
        "unique_products",
        "web_ip_count",
        "remote_access_ip_count",
        "email_ip_count",
        "proxy_ip_count",
        "network_ip_count",
        "database_ip_count",
        "iot_ip_count",
        "container_ip_count",
    ]
].sort_values(
    "account_score",
    ascending=False
)

,normalized_organization,organization_type,account_score,observation_count,unique_ips,unique_ports,unique_products,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count,database_ip_count,iot_ip_count,container_ip_count
16413,korea telecom,ISP,93.99,20646,20537,576,175,1214,95,5,3,219,21,3090,2
14403,139 162 0 0/16,UNKNOWN,88.42,919,153,145,23,56,38,1,2,1,1,0,1
16410,chunghwa telecom,ISP,87.18,3157,3027,244,174,603,76,6,14,32,1,258,0
16443,internet utilities europe and asia,UNKNOWN,86.93,2159,1946,902,30,93,37,2,18,1,2,0,2
2,aliyun computing,CLOUD_PROVIDER,86.50,71403,59465,7033,141,3754,7502,5,4760,107,75,0,11
20492,oracle,UNKNOWN,86.26,1489,1265,201,46,235,232,7,41,3,6,0,17
10,microsoft,CLOUD_PROVIDER,85.33,17946,10379,533,87,2563,224,3,90,45,34,0,45
6161,google,UNKNOWN,85.26,667714,248875,17342,76,1451,802,5,59,4,47,0,509
6174,v tal,UNKNOWN,85.15,514,464,156,35,37,13,1,45,36,1,1,0
6230,scaleway,UNKNOWN,84.96,3650,410,219,26,68,27,5,17,1,3,0,40


In [88]:
eval_labels = [
    ("korea telecom", 0, "ISP/infrastructure provider"),
    ("139 162 0 0/16", 0, "Network allocation style entity; insufficient direct-company evidence"),
    ("chunghwa telecom", 0, "ISP/infrastructure provider"),
    ("internet utilities europe and asia", 0, "Network/infrastructure organization"),
    ("aliyun computing", 0, "Cloud infrastructure provider"),
    ("oracle", 1, "Large technology organization with substantial infrastructure and multiple technology surfaces"),
    ("microsoft", 0, "Cloud/platform provider in this dataset"),
    ("google", 0, "Cloud/platform/infrastructure provider in this dataset"),
    ("ucloud", 0, "Cloud infrastructure provider"),
    ("h4y technologies", 1, "Unknown organization with meaningful multi-technology exposure"),
    ("serverfreak technologies sdn bhd", 0, "Hosting/infrastructure-oriented organization"),
    ("vodafone ono s a", 0, "ISP/telecom"),
    ("madgenius com", 1, "Unknown organization with multiple observed technology surfaces"),
    ("friendhosting", 0, "Hosting provider"),
    ("hostdime com", 0, "Hosting/infrastructure provider"),
    ("site - soluc em inf telec e eng ltda - brsite", 0, "Single observation; insufficient evidence"),
    ("cloudnet cable and telecoms", 0, "Telecom/ISP"),
    ("reassign to intriplex ( thailand )", 0, "Network allocation/reassignment entity"),
    ("rural telecom sl", 0, "Telecom/ISP"),
    ("entplexit", 0, "Single observation; insufficient evidence"),
    ("palo alto networks", 0, "Security vendor/provider"),
    ("zscaler", 0, "Security vendor/provider"),
    ("fortinet", 0, "Security vendor/provider"),
    ("zscaler australia pty", 0, "Security vendor/provider"),
    ("zscaler softech india", 0, "Security vendor/provider"),
    ("v tal", 1, "Unknown organization with broad technology exposure"),
    ("scaleway", 0, "Infrastructure/cloud-oriented organization"),
    ("tt dotcom sdn bhd", 0, "Telecom/network-oriented organization"),
    ("huawei public cloud service (huawei software t...", 0, "Cloud infrastructure provider"),
    ("peg tech", 1, "Unknown organization with significant infrastructure/technology exposure"),
]

In [89]:
len(eval_labels)

30

In [90]:
scored_accounts[
    scored_accounts["normalized_organization"].str.contains(
        "huawei",
        case=False,
        na=False
    )
][["normalized_organization", "organization_type", "account_score"]]

,normalized_organization,organization_type,account_score
28,huawei cloud brazil region,UNKNOWN,43.11
157,huawei south africa clouds,UNKNOWN,32.27
292,huawei cloud southafrica region,UNKNOWN,21.86
375,huawei cloud chile region,UNKNOWN,30.19
405,huawei cloud hongkong region,UNKNOWN,31.49
434,huawei-cloud-tr,UNKNOWN,32.76
2073,huawei international pte,UNKNOWN,51.82
2160,huawei singapore clouds,UNKNOWN,31.85
2193,huawei clouds ireland,UNKNOWN,22.44
4235,huawei-cloud-de,UNKNOWN,28.95


In [91]:
import json
from pathlib import Path

eval_labels = [
    ("korea telecom", 0, "ISP/infrastructure provider"),
    ("139 162 0 0/16", 0, "Network allocation style entity; insufficient direct-company evidence"),
    ("chunghwa telecom", 0, "ISP/infrastructure provider"),
    ("internet utilities europe and asia", 0, "Network/infrastructure organization"),
    ("aliyun computing", 0, "Cloud infrastructure provider"),
    ("oracle", 1, "Technology organization with substantial infrastructure and multiple technology surfaces"),
    ("microsoft", 0, "Cloud/platform infrastructure provider"),
    ("google", 0, "Cloud/platform/infrastructure provider"),
    ("ucloud", 0, "Cloud infrastructure provider"),
    ("h4y technologies", 1, "Unknown organization with meaningful multi-technology exposure"),
    ("serverfreak technologies sdn bhd", 0, "Hosting/infrastructure-oriented organization"),
    ("vodafone ono s a", 0, "ISP/telecom"),
    ("madgenius com", 1, "Unknown organization with multiple observed technology surfaces"),
    ("friendhosting", 0, "Hosting provider"),
    ("hostdime com", 0, "Hosting/infrastructure provider"),
    ("site - soluc em inf telec e eng ltda - brsite", 0, "Single observation; insufficient evidence"),
    ("cloudnet cable and telecoms", 0, "Telecom/ISP"),
    ("reassign to intriplex ( thailand )", 0, "Network allocation/reassignment entity"),
    ("rural telecom sl", 0, "Telecom/ISP"),
    ("entplexit", 0, "Single observation; insufficient evidence"),
    ("palo alto networks", 0, "Security vendor/provider"),
    ("zscaler", 0, "Security vendor/provider"),
    ("fortinet", 0, "Security vendor/provider"),
    ("zscaler australia pty", 0, "Security vendor/provider"),
    ("zscaler softech india", 0, "Security vendor/provider"),
    ("v tal", 1, "Unknown organization with broad technology exposure"),
    ("scaleway", 0, "Infrastructure/cloud-oriented organization"),
    ("tt dotcom sdn bhd", 0, "Telecom/network-oriented organization"),
    ("huawei public cloud service (huawei software t...", 0, "Public cloud infrastructure provider"),
    ("peg tech", 1, "Unknown organization with significant infrastructure/technology exposure"),
]

print("Number of evaluation labels:", len(eval_labels))

Number of evaluation labels: 30


In [92]:
eval_path = Path("../evals/account_relevance.jsonl")

with eval_path.open("w", encoding="utf-8") as f:
    for account, label, reason in eval_labels:
        record = {
            "account": account,
            "label": label,
            "reason": reason
        }
        f.write(json.dumps(record) + "\n")

print(f"Saved: {eval_path}")

Saved: ../evals/account_relevance.jsonl


In [93]:
with eval_path.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

print("Records:", len(records))
print("Relevant:", sum(r["label"] == 1 for r in records))
print("Not relevant:", sum(r["label"] == 0 for r in records))

Records: 30
Relevant: 5
Not relevant: 25


In [94]:
positive_pool = scored_accounts[
    (scored_accounts["organization_type"] == "UNKNOWN") &
    (
        (scored_accounts["unique_ips"] >= 10) |
        (scored_accounts["unique_products"] >= 5)
    ) &
    (
        (scored_accounts["remote_access_ip_count"] > 0) |
        (scored_accounts["database_ip_count"] > 0) |
        (scored_accounts["iot_ip_count"] > 0) |
        (scored_accounts["container_ip_count"] > 0) |
        (scored_accounts["web_ip_count"] > 0)
    )
].sort_values(
    "account_score",
    ascending=False
)

positive_pool[
    [
        "normalized_organization",
        "account_score",
        "observation_count",
        "unique_ips",
        "unique_products",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
        "container_ip_count",
    ]
].head(30)

,normalized_organization,account_score,observation_count,unique_ips,unique_products,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count,container_ip_count
14403,139 162 0 0/16,88.42,919,153,23,56,38,1,0,1
16443,internet utilities europe and asia,86.93,2159,1946,30,93,37,2,0,2
20492,oracle,86.26,1489,1265,46,235,232,6,0,17
6161,google,85.26,667714,248875,76,1451,802,47,0,509
6174,v tal,85.15,514,464,35,37,13,1,1,0
6230,scaleway,84.96,3650,410,26,68,27,3,0,40
16790,tt dotcom sdn bhd,83.16,119,110,20,9,4,1,4,0
22529,huawei public cloud service (huawei software t...,80.79,1580,1198,37,114,75,9,0,0
16435,internet utilities na,80.50,501,486,26,108,19,1,0,1
16544,ee,80.33,606,540,27,125,92,2,2,2


In [95]:
additional_candidates = scored_accounts[
    scored_accounts["normalized_organization"].isin([
        "oracle",
        "v tal",
        "ncr",
        "iomart managed services",
        "ace data centers ii l l c",
        "newfold digital",
        "elisa oyj",
        "rcs & rds residential",
        "fastweb spa",
        "the constant",
    ])
][
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "unique_ports",
        "unique_products",
        "web_ip_count",
        "remote_access_ip_count",
        "email_ip_count",
        "proxy_ip_count",
        "network_ip_count",
        "database_ip_count",
        "iot_ip_count",
        "container_ip_count",
    ]
].sort_values(
    "account_score",
    ascending=False
)

additional_candidates

,normalized_organization,organization_type,account_score,observation_count,unique_ips,unique_ports,unique_products,web_ip_count,remote_access_ip_count,email_ip_count,proxy_ip_count,network_ip_count,database_ip_count,iot_ip_count,container_ip_count
20492,oracle,UNKNOWN,86.26,1489,1265,201,46,235,232,7,41,3,6,0,17
6174,v tal,UNKNOWN,85.15,514,464,156,35,37,13,1,45,36,1,1,0
20686,ace data centers ii l l c,UNKNOWN,79.50,286,266,131,16,21,38,3,5,1,1,0,2
16422,newfold digital,UNKNOWN,79.32,868,682,136,18,181,29,33,3,0,11,0,1
14711,elisa oyj,UNKNOWN,79.22,182,148,152,18,3,2,1,1,1,1,4,0
22619,ncr,UNKNOWN,79.17,391,341,273,19,14,22,3,3,2,1,0,0
20488,the constant,UNKNOWN,78.23,2962,917,305,26,574,69,12,18,1,0,0,2
20506,rcs & rds residential,UNKNOWN,77.35,204,187,67,27,20,2,0,1,3,1,33,1
481,iomart managed services,UNKNOWN,76.92,97,63,41,17,23,3,3,1,2,1,0,0
14362,fastweb spa,UNKNOWN,76.65,252,245,60,32,28,4,3,1,41,0,3,0


In [96]:
additional_labels = [
    ("oracle", 1, "Technology organization with substantial infrastructure and multiple technology surfaces"),
    ("v tal", 1, "Unknown organization with broad technology and network exposure"),
    ("ace data centers ii l l c", 0, "Data-center/infrastructure provider"),
    ("newfold digital", 0, "Hosting/web infrastructure provider"),
    ("elisa oyj", 0, "Telecom/network provider"),
    ("ncr", 1, "Enterprise technology organization with broad technology exposure"),
    ("the constant", 1, "Unknown organization with substantial remote-access and infrastructure exposure"),
    ("rcs & rds residential", 0, "Residential/network infrastructure provider"),
    ("iomart managed services", 0, "Managed infrastructure/cloud services provider"),
    ("fastweb spa", 0, "Telecom/network provider"),
]

In [97]:
with eval_path.open("a", encoding="utf-8") as f:
    for account, label, reason in additional_labels:
        record = {
            "account": account,
            "label": label,
            "reason": reason
        }
        f.write(json.dumps(record) + "\n")

print("Added:", len(additional_labels))

Added: 10


In [98]:
with eval_path.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

print("Records:", len(records))
print("Relevant:", sum(r["label"] == 1 for r in records))
print("Not relevant:", sum(r["label"] == 0 for r in records))

Records: 40
Relevant: 9
Not relevant: 31


In [109]:
import pandas as pd
import json

# Load evaluation labels
with eval_path.open("r", encoding="utf-8") as f:
    eval_df = pd.DataFrame(
        [json.loads(line) for line in f]
    )

# Keep only the fields we need from scored_accounts
baseline = scored_accounts[
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "unique_ports",
        "unique_products",
        "web_ip_count",
        "remote_access_ip_count",
        "email_ip_count",
        "proxy_ip_count",
        "network_ip_count",
        "database_ip_count",
        "iot_ip_count",
        "container_ip_count",
    ]
].merge(
    eval_df,
    left_on="normalized_organization",
    right_on="account",
    how="inner"
)

baseline = baseline.sort_values(
    "account_score",
    ascending=False
).reset_index(drop=True)

print("Evaluation accounts matched:", len(baseline))

# Top 10 predictions
top_10 = baseline.head(10)

top_10[
    [
        "normalized_organization",
        "account_score",
        "label",
        "reason"
    ]
]

Evaluation accounts matched: 39


,normalized_organization,account_score,label,reason
0,korea telecom,93.99,0,ISP/infrastructure provider
1,139 162 0 0/16,88.42,0,Network allocation style entity; insufficient direct-company evidence
2,chunghwa telecom,87.18,0,ISP/infrastructure provider
3,internet utilities europe and asia,86.93,0,Network/infrastructure organization
4,aliyun computing,86.50,0,Cloud infrastructure provider
5,oracle,86.26,1,Technology organization with substantial infrastructure and multiple technology surfaces
6,oracle,86.26,1,Technology organization with substantial infrastructure and multiple technology surfaces
7,microsoft,85.33,0,Cloud/platform infrastructure provider
8,google,85.26,0,Cloud/platform/infrastructure provider
9,v tal,85.15,1,Unknown organization with broad technology exposure


In [100]:
top_k = 10

predicted_positive = set(
    baseline.head(top_k)["normalized_organization"]
)

actual_positive = set(
    baseline[baseline["label"] == 1]["normalized_organization"]
)

true_positives = len(
    predicted_positive & actual_positive
)

false_positives = len(
    predicted_positive - actual_positive
)

false_negatives = len(
    actual_positive - predicted_positive
)

precision = (
    true_positives /
    (true_positives + false_positives)
    if (true_positives + false_positives) > 0
    else 0
)

recall = (
    true_positives /
    (true_positives + false_negatives)
    if (true_positives + false_negatives) > 0
    else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print(f"Top-K: {top_k}")
print(f"True Positives: {true_positives}")
print(f"False Positives: {false_positives}")
print(f"False Negatives: {false_negatives}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")

Top-K: 10
True Positives: 2
False Positives: 7
False Negatives: 5
Precision: 0.222
Recall: 0.286
F1: 0.250


In [110]:
eval_positive_check = baseline[
    baseline["label"] == 1
][
    [
        "normalized_organization",
        "organization_type",
        "account_score",
        "observation_count",
        "unique_ips",
        "unique_products",
        "web_ip_count",
        "remote_access_ip_count",
        "database_ip_count",
        "iot_ip_count",
        "container_ip_count",
    ]
].sort_values(
    "account_score",
    ascending=False
)

eval_positive_check

,normalized_organization,organization_type,account_score,observation_count,unique_ips,unique_products,web_ip_count,remote_access_ip_count,database_ip_count,iot_ip_count,container_ip_count
5,oracle,UNKNOWN,86.26,1489,1265,46,235,232,6,0,17
6,oracle,UNKNOWN,86.26,1489,1265,46,235,232,6,0,17
9,v tal,UNKNOWN,85.15,514,464,35,37,13,1,1,0
10,v tal,UNKNOWN,85.15,514,464,35,37,13,1,1,0
13,peg tech,UNKNOWN,80.22,1026,1005,31,508,37,12,0,0
17,ncr,UNKNOWN,79.17,391,341,19,14,22,1,0,0
18,the constant,UNKNOWN,78.23,2962,917,26,574,69,0,0,2
22,h4y technologies,UNKNOWN,57.97,138,132,12,22,6,0,0,0
23,madgenius com,UNKNOWN,57.08,18,17,7,4,1,0,0,0


In [104]:
print("Evaluation accounts matched:", len(baseline))

Evaluation accounts matched: 39


In [105]:
matched_accounts = set(baseline["normalized_organization"])
eval_accounts = set(eval_df["account"])

missing_accounts = eval_accounts - matched_accounts

print("Missing accounts:")
for account in missing_accounts:
    print(repr(account))

Missing accounts:
'huawei public cloud service (huawei software t...'


In [106]:
scored_accounts[
    scored_accounts["normalized_organization"].str.contains(
        "huawei public cloud service",
        case=False,
        na=False
    )
][[
    "normalized_organization",
    "organization_type",
    "account_score"
]]

,normalized_organization,organization_type,account_score
6536,huawei public cloud service (huawei software t...,UNKNOWN,51.94
22529,huawei public cloud service (huawei software t...,UNKNOWN,80.79


In [107]:
huawei_accounts = scored_accounts[
    scored_accounts["normalized_organization"].str.contains(
        "huawei public cloud service",
        case=False,
        na=False
    )
][[
    "normalized_organization",
    "organization_type",
    "account_score",
    "observation_count",
    "unique_ips"
]].sort_values(
    "account_score",
    ascending=False
)

pd.set_option("display.max_colwidth", None)

huawei_accounts

,normalized_organization,organization_type,account_score,observation_count,unique_ips
22529,huawei public cloud service (huawei software technologies ltd co),UNKNOWN,80.79,1580,1198
6536,huawei public cloud service (huawei software technologies co ltd),UNKNOWN,51.94,36,35


In [108]:
eval_df.loc[
    eval_df["account"].str.startswith(
        "huawei public cloud service",
        na=False
    ),
    "account"
] = "huawei public cloud service (huawei software technologies ltd co)"

In [111]:
matched_accounts = set(baseline["normalized_organization"])
eval_accounts = set(eval_df["account"])

print("Evaluation accounts:", len(eval_accounts))
print("Matched accounts:", len(matched_accounts))
print("\nMissing account(s):")

for account in sorted(eval_accounts - matched_accounts):
    print(repr(account))

Evaluation accounts: 38
Matched accounts: 37

Missing account(s):
'huawei public cloud service (huawei software t...'


In [113]:
with eval_path.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

print("JSONL records:", len(records))

for i, record in enumerate(records, start=1):
    print(i, record["account"])

JSONL records: 40
1 korea telecom
2 139 162 0 0/16
3 chunghwa telecom
4 internet utilities europe and asia
5 aliyun computing
6 oracle
7 microsoft
8 google
9 ucloud
10 h4y technologies
11 serverfreak technologies sdn bhd
12 vodafone ono s a
13 madgenius com
14 friendhosting
15 hostdime com
16 site - soluc em inf telec e eng ltda - brsite
17 cloudnet cable and telecoms
18 reassign to intriplex ( thailand )
19 rural telecom sl
20 entplexit
21 palo alto networks
22 zscaler
23 fortinet
24 zscaler australia pty
25 zscaler softech india
26 v tal
27 scaleway
28 tt dotcom sdn bhd
29 huawei public cloud service (huawei software t...
30 peg tech
31 oracle
32 v tal
33 ace data centers ii l l c
34 newfold digital
35 elisa oyj
36 ncr
37 the constant
38 rcs & rds residential
39 iomart managed services
40 fastweb spa


In [114]:
for i, record in enumerate(records, start=1):
    print(
        i,
        "|",
        record["account"],
        "| label =", record["label"],
        "|",
        record["reason"]
    )

1 | korea telecom | label = 0 | ISP/infrastructure provider
2 | 139 162 0 0/16 | label = 0 | Network allocation style entity; insufficient direct-company evidence
3 | chunghwa telecom | label = 0 | ISP/infrastructure provider
4 | internet utilities europe and asia | label = 0 | Network/infrastructure organization
5 | aliyun computing | label = 0 | Cloud infrastructure provider
6 | oracle | label = 1 | Technology organization with substantial infrastructure and multiple technology surfaces
7 | microsoft | label = 0 | Cloud/platform infrastructure provider
8 | google | label = 0 | Cloud/platform/infrastructure provider
9 | ucloud | label = 0 | Cloud infrastructure provider
10 | h4y technologies | label = 1 | Unknown organization with meaningful multi-technology exposure
11 | serverfreak technologies sdn bhd | label = 0 | Hosting/infrastructure-oriented organization
12 | vodafone ono s a | label = 0 | ISP/telecom
13 | madgenius com | label = 1 | Unknown organization with multiple observed

In [115]:
print("Total rows:", len(records))
print("Unique accounts:", len(set(r["account"] for r in records)))

from collections import Counter

counts = Counter(r["account"] for r in records)

print("\nDuplicates:")
for account, count in counts.items():
    if count > 1:
        print(account, "->", count)

Total rows: 40
Unique accounts: 38

Duplicates:
oracle -> 2
v tal -> 2


In [116]:
import json

final_eval_labels = [
    {
        "account": "korea telecom",
        "label": 0,
        "reason": "ISP/infrastructure provider"
    },
    {
        "account": "139 162 0 0/16",
        "label": 0,
        "reason": "Network allocation style entity; insufficient direct-company evidence"
    },
    {
        "account": "chunghwa telecom",
        "label": 0,
        "reason": "ISP/infrastructure provider"
    },
    {
        "account": "internet utilities europe and asia",
        "label": 0,
        "reason": "Network/infrastructure organization"
    },
    {
        "account": "aliyun computing",
        "label": 0,
        "reason": "Cloud infrastructure provider"
    },
    {
        "account": "oracle",
        "label": 1,
        "reason": "Technology organization with substantial infrastructure and multiple technology surfaces"
    },
    {
        "account": "microsoft",
        "label": 0,
        "reason": "Cloud/platform infrastructure provider"
    },
    {
        "account": "google",
        "label": 0,
        "reason": "Cloud/platform/infrastructure provider"
    },
    {
        "account": "ucloud",
        "label": 0,
        "reason": "Cloud infrastructure provider"
    },
    {
        "account": "h4y technologies",
        "label": 1,
        "reason": "Unknown organization with meaningful multi-technology exposure"
    },
    {
        "account": "serverfreak technologies sdn bhd",
        "label": 0,
        "reason": "Hosting/infrastructure-oriented organization"
    },
    {
        "account": "vodafone ono s a",
        "label": 0,
        "reason": "ISP/telecom"
    },
    {
        "account": "madgenius com",
        "label": 1,
        "reason": "Unknown organization with multiple observed technology surfaces"
    },
    {
        "account": "friendhosting",
        "label": 0,
        "reason": "Hosting provider"
    },
    {
        "account": "hostdime com",
        "label": 0,
        "reason": "Hosting/infrastructure provider"
    },
    {
        "account": "site - soluc em inf telec e eng ltda - brsite",
        "label": 0,
        "reason": "Single observation; insufficient evidence"
    },
    {
        "account": "cloudnet cable and telecoms",
        "label": 0,
        "reason": "Telecom/ISP"
    },
    {
        "account": "reassign to intriplex ( thailand )",
        "label": 0,
        "reason": "Network allocation/reassignment entity"
    },
    {
        "account": "rural telecom sl",
        "label": 0,
        "reason": "Telecom/ISP"
    },
    {
        "account": "entplexit",
        "label": 0,
        "reason": "Single observation; insufficient evidence"
    },
    {
        "account": "palo alto networks",
        "label": 0,
        "reason": "Security vendor/provider"
    },
    {
        "account": "zscaler",
        "label": 0,
        "reason": "Security vendor/provider"
    },
    {
        "account": "fortinet",
        "label": 0,
        "reason": "Security vendor/provider"
    },
    {
        "account": "zscaler australia pty",
        "label": 0,
        "reason": "Security vendor/provider"
    },
    {
        "account": "zscaler softech india",
        "label": 0,
        "reason": "Security vendor/provider"
    },
    {
        "account": "v tal",
        "label": 1,
        "reason": "Unknown organization with broad technology exposure"
    },
    {
        "account": "scaleway",
        "label": 0,
        "reason": "Infrastructure/cloud-oriented provider"
    },
    {
        "account": "tt dotcom sdn bhd",
        "label": 0,
        "reason": "Telecom/network provider"
    },
    {
        "account": "huawei public cloud service (huawei software technologies ltd co)",
        "label": 0,
        "reason": "Public cloud infrastructure provider"
    },
    {
        "account": "peg tech",
        "label": 1,
        "reason": "Unknown organization with significant infrastructure and technology exposure"
    }
]

eval_path = Path("../evals/account_relevance.jsonl")

with eval_path.open("w", encoding="utf-8") as f:
    for record in final_eval_labels:
        f.write(json.dumps(record) + "\n")

print("Wrote:", len(final_eval_labels), "evaluation records")
print("Unique accounts:", len(set(r["account"] for r in final_eval_labels)))
print("Positive:", sum(r["label"] == 1 for r in final_eval_labels))
print("Negative:", sum(r["label"] == 0 for r in final_eval_labels))

Wrote: 30 evaluation records
Unique accounts: 30
Positive: 5
Negative: 25


In [118]:
import pandas as pd

eval_df = pd.read_json(
    eval_path,
    lines=True
)

print("Evaluation rows:", len(eval_df))
print("Positive labels:", eval_df["label"].sum())
print("Negative labels:", (eval_df["label"] == 0).sum())

baseline = eval_df.merge(
    scored_accounts[
        [
            "normalized_organization",
            "account_score",
            "organization_type"
        ]
    ],
    left_on="account",
    right_on="normalized_organization",
    how="inner"
)

print("\nMatched rows:", len(baseline))
print("Unique matched accounts:", baseline["account"].nunique())

print("\nTop 10 baseline accounts:")
print(
    baseline
    .sort_values("account_score", ascending=False)
    [
        [
            "account",
            "label",
            "account_score",
            "organization_type"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

Evaluation rows: 30
Positive labels: 5
Negative labels: 25

Matched rows: 30
Unique matched accounts: 30

Top 10 baseline accounts:
                           account  label  account_score organization_type
                     korea telecom      0          93.99               ISP
                    139 162 0 0/16      0          88.42           UNKNOWN
                  chunghwa telecom      0          87.18               ISP
internet utilities europe and asia      0          86.93           UNKNOWN
                  aliyun computing      0          86.50    CLOUD_PROVIDER
                            oracle      1          86.26           UNKNOWN
                         microsoft      0          85.33    CLOUD_PROVIDER
                            google      0          85.26           UNKNOWN
                             v tal      1          85.15           UNKNOWN
                          scaleway      0          84.96           UNKNOWN


In [119]:
import numpy as np

# Rank all 30 evaluation accounts by the current baseline score
ranked = baseline.sort_values(
    "account_score",
    ascending=False
).reset_index(drop=True)

# Top-K prediction
K = 10

predicted_positive = set(
    ranked.head(K)["account"]
)

actual_positive = set(
    ranked.loc[ranked["label"] == 1, "account"]
)

true_positive = len(predicted_positive & actual_positive)
false_positive = len(predicted_positive - actual_positive)
false_negative = len(actual_positive - predicted_positive)

precision = (
    true_positive / (true_positive + false_positive)
    if true_positive + false_positive > 0
    else 0
)

recall = (
    true_positive / (true_positive + false_negative)
    if true_positive + false_negative > 0
    else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0
    else 0
)

print("Baseline Evaluation")
print("-------------------")
print("K:", K)
print("True positives:", true_positive)
print("False positives:", false_positive)
print("False negatives:", false_negative)
print(f"Precision@{K}: {precision:.3f}")
print(f"Recall@{K}:    {recall:.3f}")
print(f"F1@{K}:        {f1:.3f}")

Baseline Evaluation
-------------------
K: 10
True positives: 2
False positives: 8
False negatives: 3
Precision@10: 0.200
Recall@10:    0.400
F1@10:        0.267


In [120]:
print([
    column
    for column in scored_accounts.columns
    if column in [
        "normalized_organization",
        "organization_type",
        "unique_ips",
        "unique_ports",
        "unique_products",
        "exposure_signal_count",
        "is_infrastructure_provider",
        "is_security_provider",
        "small_infrastructure",
        "medium_infrastructure",
        "large_infrastructure",
        "has_multiple_technology_surfaces",
    ]
])

['normalized_organization', 'organization_type', 'unique_ips', 'unique_ports', 'unique_products']


In [122]:
from src.signals.account_signals import build_account_signals

scored_accounts_signals = build_account_signals(scored_accounts)

print([
    column
    for column in scored_accounts_signals.columns
    if column.startswith("has_")
    or column in [
        "exposure_signal_count",
        "technology_diversity",
        "infrastructure_scale",
    ]
])

['has_web_exposure', 'has_remote_access_exposure', 'has_email_exposure', 'has_proxy_exposure', 'has_network_exposure', 'exposure_signal_count', 'technology_diversity', 'infrastructure_scale']


In [123]:
from src.scoring.icp import add_icp_features

scored_accounts_icp = add_icp_features(
    scored_accounts_signals
)

print([
    column
    for column in scored_accounts_icp.columns
    if column in [
        "normalized_organization",
        "organization_type",
        "unique_ips",
        "unique_ports",
        "unique_products",
        "exposure_signal_count",
        "is_infrastructure_provider",
        "is_security_provider",
        "small_infrastructure",
        "medium_infrastructure",
        "large_infrastructure",
        "has_multiple_technology_surfaces",
    ]
])


['normalized_organization', 'organization_type', 'unique_ips', 'unique_ports', 'unique_products', 'exposure_signal_count', 'is_infrastructure_provider', 'is_security_provider', 'small_infrastructure', 'medium_infrastructure', 'large_infrastructure', 'has_multiple_technology_surfaces']


In [124]:
from src.scoring.icp_score import add_icp_score

icp_scored_accounts = add_icp_score(
    scored_accounts_icp
)

print(
    icp_scored_accounts[
        [
            "normalized_organization",
            "organization_type",
            "exposure_signal_count",
            "unique_products",
            "unique_ips",
            "icp_fit_score",
        ]
    ]
    .sort_values("icp_fit_score", ascending=False)
    .head(15)
    .to_string(index=False)
)

          normalized_organization organization_type  exposure_signal_count  unique_products  unique_ips  icp_fit_score
                   telia norge as           UNKNOWN                      3                9          24           70.0
             volonet technologies           UNKNOWN                      3                9          43           70.0
                        sharktech           UNKNOWN                      3               11         120           70.0
                          timeweb           UNKNOWN                      4               10          89           70.0
                         readyidc           UNKNOWN                      4               14          50           70.0
            mass response service           UNKNOWN                      3                7          14           70.0
        grupo loading systems s l           UNKNOWN                      3                5          37           70.0
                sitkom spol s r o           UNKN

In [125]:
from importlib import reload
import src.scoring.icp_score as icp_score

reload(icp_score)

icp_scored_accounts = icp_score.add_icp_score(
    scored_accounts_icp
)

print(
    icp_scored_accounts[
        [
            "normalized_organization",
            "organization_type",
            "exposure_signal_count",
            "unique_products",
            "unique_ips",
            "icp_fit_score",
        ]
    ]
    .sort_values("icp_fit_score", ascending=False)
    .head(15)
    .to_string(index=False)
)

                                          normalized_organization organization_type  exposure_signal_count  unique_products  unique_ips  icp_fit_score
huawei public cloud service (huawei software technologies ltd co)           UNKNOWN                      5               37        1198           85.0
                                                         peg tech           UNKNOWN                      5               31        1005           85.0
                                                              gtt           UNKNOWN                      5               22        1358           85.0
                               internet utilities europe and asia           UNKNOWN                      5               30        1946           85.0
                                                    viettel group           UNKNOWN                      5               60         860           85.0
                                                  sakura internet           UNKNOWN           

In [127]:
# Merge the ICP score with our labelled evaluation set

icp_eval = eval_df.merge(
    icp_scored_accounts[
        [
            "normalized_organization",
            "icp_fit_score",
            "organization_type"
        ]
    ],
    left_on="account",
    right_on="normalized_organization",
    how="inner"
)

print("Evaluation rows:", len(icp_eval))
print("Unique accounts:", icp_eval["account"].nunique())


# Rank by ICP score
ranked_icp = (
    icp_eval
    .sort_values("icp_fit_score", ascending=False)
    .reset_index(drop=True)
)

K = 10

predicted_positive = set(
    ranked_icp.head(K)["account"]
)

actual_positive = set(
    ranked_icp.loc[
        ranked_icp["label"] == 1,
        "account"
    ]
)

true_positive = len(
    predicted_positive & actual_positive
)

false_positive = len(
    predicted_positive - actual_positive
)

false_negative = len(
    actual_positive - predicted_positive
)

precision = (
    true_positive /
    (true_positive + false_positive)
    if true_positive + false_positive > 0
    else 0
)

recall = (
    true_positive /
    (true_positive + false_negative)
    if true_positive + false_negative > 0
    else 0
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall > 0
    else 0
)

print("\nICP-aware Evaluation")
print("--------------------")
print("K:", K)
print("True positives:", true_positive)
print("False positives:", false_positive)
print("False negatives:", false_negative)
print(f"Precision@{K}: {precision:.3f}")
print(f"Recall@{K}:    {recall:.3f}")
print(f"F1@{K}:        {f1:.3f}")

print("\nTop 10:")
print(
    ranked_icp.head(10)[
        [
            "account",
            "label",
            "icp_fit_score",
            "organization_type"
        ]
    ].to_string(index=False)
)

Evaluation rows: 30
Unique accounts: 30

ICP-aware Evaluation
--------------------
K: 10
True positives: 5
False positives: 5
False negatives: 0
Precision@10: 0.500
Recall@10:    1.000
F1@10:        0.667

Top 10:
                                                          account  label  icp_fit_score organization_type
                                                         peg tech      1           85.0           UNKNOWN
                               internet utilities europe and asia      0           85.0           UNKNOWN
huawei public cloud service (huawei software technologies ltd co)      0           85.0           UNKNOWN
                                                           oracle      1           85.0           UNKNOWN
                                                            v tal      1           83.0           UNKNOWN
                                                tt dotcom sdn bhd      0           83.0           UNKNOWN
                                            

In [129]:
ranked_icp = (
    eval_df
    .merge(
        icp_scored_accounts[
            [
                "normalized_organization",
                "icp_fit_score",
                "organization_type",
                "exposure_signal_count",
                "unique_products",
                "unique_ips",
            ]
        ],
        left_on="account",
        right_on="normalized_organization",
        how="inner",
    )
    .sort_values(
        ["icp_fit_score", "exposure_signal_count", "unique_products", "unique_ips"],
        ascending=[False, False, False, False],
    )
    .reset_index(drop=True)
)

print(
    ranked_icp[
        [
            "account",
            "label",
            "icp_fit_score",
            "organization_type",
            "exposure_signal_count",
            "unique_products",
            "unique_ips",
        ]
    ].to_string(index=False)
)

                                                          account  label  icp_fit_score organization_type  exposure_signal_count  unique_products  unique_ips
                                                           oracle      1           85.0           UNKNOWN                      5               46        1265
huawei public cloud service (huawei software technologies ltd co)      0           85.0           UNKNOWN                      5               37        1198
                                                         peg tech      1           85.0           UNKNOWN                      5               31        1005
                               internet utilities europe and asia      0           85.0           UNKNOWN                      5               30        1946
                                                            v tal      1           83.0           UNKNOWN                      5               35         464
                                                    

In [130]:
from pathlib import Path
import json

results = {
    "evaluation_set": "account_relevance.jsonl",
    "evaluation_size": 30,
    "positive_accounts": 5,
    "negative_accounts": 25,
    "k": 10,
    "baseline": {
        "true_positive": 2,
        "false_positive": 8,
        "false_negative": 3,
        "precision_at_10": 0.200,
        "recall_at_10": 0.400,
        "f1_at_10": 0.267,
    },
    "icp_aware": {
        "true_positive": 5,
        "false_positive": 5,
        "false_negative": 0,
        "precision_at_10": 0.500,
        "recall_at_10": 1.000,
        "f1_at_10": 0.667,
    },
    "notes": [
        "Labels are hand-authored proxy relevance judgements.",
        "Labels do not represent observed purchase intent or CRM outcomes.",
        "The evaluation set is small and should be treated as directional.",
    ],
}

results_path = Path("../evals/results.json")

with results_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Saved evaluation results to: {results_path}")

Saved evaluation results to: ../evals/results.json


In [131]:
oracle = icp_scored_accounts[
    icp_scored_accounts["normalized_organization"] == "oracle"
]

print(
    oracle[
        [
            "normalized_organization",
            "organization_type",
            "observation_count",
            "unique_ips",
            "unique_ports",
            "unique_products",
            "countries_observed",
            "first_seen",
            "last_seen",
            "web_ip_count",
            "remote_access_ip_count",
            "email_ip_count",
            "proxy_ip_count",
            "network_ip_count",
            "database_ip_count",
            "iot_ip_count",
            "container_ip_count",
            "security_network_ip_count",
            "file_transfer_ip_count",
            "exposure_signal_count",
            "icp_fit_score",
        ]
    ].to_string(index=False)
)

normalized_organization organization_type  observation_count  unique_ips  unique_ports  unique_products  countries_observed                 first_seen                  last_seen  web_ip_count  remote_access_ip_count  email_ip_count  proxy_ip_count  network_ip_count  database_ip_count  iot_ip_count  container_ip_count  security_network_ip_count  file_transfer_ip_count  exposure_signal_count  icp_fit_score
                 oracle           UNKNOWN               1489        1265           201               46                  23 2026-09-14T09:55:02.515602 2026-09-14T10:18:25.784420           235                     232               7              41                 3                  6             0                  17                          0                       2                      5           85.0
